# FLOPS and Inference Time Benchmark for ChronosBoltPipelineWithRetrieval Models (v2)

This notebook measures FLOPS and inference time for 6 ablation model variants.
Uses actual retrieval data from ETTh1 dataset using retrieve_X_norm.

In [7]:
import sys
import os
# Add parent directory to path
sys.path.insert(0, os.path.abspath('..'))
# Also add ts-rag directory if it exists
if os.path.exists('../ts-rag'):
    sys.path.insert(0, os.path.abspath('../ts-rag'))

import torch
import torch.nn as nn
import time
import numpy as np
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

# FLOPS measurement
try:
    from thop import profile, clever_format
    HAS_THOP = True
except ImportError:
    print("thop not installed. Install with: pip install thop")
    HAS_THOP = False


In [8]:
# Import retriever
from retrieve_X_norm import do_retrieve
import pandas as pd

# Dataset configuration for ETTh1
DATASET_NAME = 'ETTh1'
ROOT_DIR = '../datasets/ETT-small'
RETRIEVAL_DATABASE_DIR = '../retrieval_database/'
METRIC = 'euclidean'  # 'euclidean', 'cosine', or 'dtw'
SEED = 42

# Metadata for retrieval
metadata = {
    'database_name': ['ETTh1'],  # Use ETTh1 itself as database
    'lookback_length': CONTEXT_LENGTH,
    'frequency': 'hour',
}

print(f"Dataset: {DATASET_NAME}")
print(f"Root directory: {ROOT_DIR}")
print(f"Retrieval database directory: {RETRIEVAL_DATABASE_DIR}")
print(f"Metric: {METRIC}")

Dataset: ETTh1
Root directory: ../datasets/ETT-small
Retrieval database directory: ../retrieval_database/
Metric: euclidean


In [9]:
from transformers import T5Config

# Import models with proper package context to handle relative imports
import importlib.util
import types

# Get the absolute path to ts-rag directory
# Notebook is at /home/seunghan.lee/workspace/experiments
# Models are at /home/seunghan.lee/workspace/ts-rag/models
ts_rag_path = os.path.abspath('../ts-rag')
models_path = os.path.join(ts_rag_path, 'models')

if ts_rag_path not in sys.path:
    sys.path.insert(0, ts_rag_path)

# Ensure ts_rag.models is in sys.modules for relative imports
if 'ts_rag' not in sys.modules:
    ts_rag_module = types.ModuleType('ts_rag')
    sys.modules['ts_rag'] = ts_rag_module
    
if 'ts_rag.models' not in sys.modules:
    models_module = types.ModuleType('ts_rag.models')
    models_module.__path__ = [models_path]
    sys.modules['ts_rag.models'] = models_module

# Load dependencies in order: utils.py -> base.py
# First load utils.py (base.py depends on it)
if 'ts_rag.models.utils' not in sys.modules:
    utils_file_path = os.path.join(models_path, 'utils.py')
    if os.path.exists(utils_file_path):
        utils_spec = importlib.util.spec_from_file_location('ts_rag.models.utils', utils_file_path)
        utils_module = importlib.util.module_from_spec(utils_spec)
        utils_module.__package__ = 'ts_rag.models'
        utils_module.__file__ = utils_file_path
        sys.modules['ts_rag.models.utils'] = utils_module
        utils_spec.loader.exec_module(utils_module)

# Then load base.py (models depend on it)
if 'ts_rag.models.base' not in sys.modules:
    base_file_path = os.path.join(models_path, 'base.py')
    base_spec = importlib.util.spec_from_file_location('ts_rag.models.base', base_file_path)
    base_module = importlib.util.module_from_spec(base_spec)
    base_module.__package__ = 'ts_rag.models'
    base_module.__file__ = base_file_path
    sys.modules['ts_rag.models.base'] = base_module
    base_spec.loader.exec_module(base_module)

# Model file names (without .py extension)
model_files = [
    'ChronosBolt_TabPFN_dualhead_R_as_key_ablation_cross_self',
    'ChronosBolt_TabPFN_dualhead_R_as_key_ablation_cross',
    'ChronosBolt_TabPFN_dualhead_R_as_key_ablation_main_cross',
    'ChronosBolt_TabPFN_dualhead_R_as_key_ablation_main_self',
    'ChronosBolt_TabPFN_dualhead_R_as_key_ablation_main',
    'ChronosBolt_TabPFN_dualhead_R_as_key_ablation_self',
]

models_list = []
pipelines_list = []

for model_file in model_files:
    # Load module as part of ts_rag.models package
    module_name = f'ts_rag.models.{model_file}'
    file_path = os.path.join(models_path, f'{model_file}.py')
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Model file not found: {file_path}")
    
    # Create spec with proper package name
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name}")
    
    module = importlib.util.module_from_spec(spec)
    # Set package attribute for relative imports to work
    module.__package__ = 'ts_rag.models'
    module.__file__ = file_path
    # Add to sys.modules so relative imports work
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    
    models_list.append(module.ChronosBoltModelForForecastingWithRetrieval)
    pipelines_list.append(module.ChronosBoltPipelineWithRetrieval)

Model1, Model2, Model3, Model4, Model5, Model6 = models_list
Pipeline1, Pipeline2, Pipeline3, Pipeline4, Pipeline5, Pipeline6 = pipelines_list

MODEL_NAMES = [
    "cross_self",
    "cross",
    "main_cross",
    "main_self",
    "main",
    "self"
]

MODELS = [Model1, Model2, Model3, Model4, Model5, Model6]
PIPELINES = [Pipeline1, Pipeline2, Pipeline3, Pipeline4, Pipeline5, Pipeline6]
# Retrieval settings
RETRIEVE_SUFFIX = 'X-norm'  # Suffix for retrieved data file
USE_ACTUAL_RETRIEVAL = True  # Set to False to use dummy data

In [17]:
# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model hyperparameters
BATCH_SIZE = 1
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 64
NUM_RETRIEVED = 8  # Number of retrieved sequences
RETRIEVED_SEQ_LENGTH = CONTEXT_LENGTH + PREDICTION_LENGTH  # 64 (context) + 64 (prediction)

# T5 base config
t5_config = T5Config(
    d_model=512,
    d_ff=2048,
    num_layers=6,
    num_decoder_layers=6,
    num_heads=8,
    vocab_size=1,
    decoder_start_token_id=0,
    dense_act_fn="gelu_new",
    dropout_rate=0.1,
    initializer_factor=1.0,
)

# Chronos config
chronos_config = {
    "context_length": CONTEXT_LENGTH,
    "prediction_length": PREDICTION_LENGTH,
    "input_patch_size": 32,
    "input_patch_stride": 16,
    "quantiles": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    "use_reg_token": False
}

t5_config.chronos_config = chronos_config

# Augment type (all models use 'moe')
augment = 'moe'

Using device: cuda


In [18]:
def load_retrieved_data(dataset_name, root_dir, context_length, prediction_length, metadata, retrieve_suffix):
    """Load retrieved data from CSV file"""
    retrieval_database_names = '_'.join(metadata['database_name'])
    suffix = retrieve_suffix if retrieve_suffix else ''
    suffix = suffix if suffix == '' else f'_{suffix}'
    
    retrieved_data_path = os.path.join(
        root_dir, 
        f'{dataset_name}_retrieve_{retrieval_database_names}_{metadata["lookback_length"]}_only_self_train_None{suffix}.csv'
    )
    
    if not os.path.exists(retrieved_data_path):
        print(f'Retrieved data not found at {retrieved_data_path}')
        print('Performing retrieval...')
        # Perform retrieval
        retrieved_df = do_retrieve(
            original_data_name=dataset_name,
            retrieval_database_dir=RETRIEVAL_DATABASE_DIR,
            root_dir=ROOT_DIR,
            metadata=metadata,
            mode='only_self_train',
            top_k=NUM_RETRIEVED,
            context_length=context_length,
            prediction_length=prediction_length,
            seed=SEED,
            metric=METRIC,
            save=True,
            retrieve_suffix=retrieve_suffix
        )
    else:
        print(f'Loading retrieved data from {retrieved_data_path}')
        retrieved_df = pd.read_csv(retrieved_data_path)
    
    return retrieved_df

In [19]:
def create_dummy_inputs(batch_size, context_length, num_retrieved, retrieved_seq_length):
    """Create dummy inputs for forward pass"""
    context = torch.randn(batch_size, context_length, device=device, dtype=torch.float32)
    retrieved_seq = torch.randn(batch_size, num_retrieved, retrieved_seq_length, device=device, dtype=torch.float32)
    distances = torch.randn(batch_size, num_retrieved, device=device, dtype=torch.float32)
    return context, retrieved_seq, distances

def measure_inference_time(model, context, retrieved_seq, distances, num_warmup=10, num_runs=100):
    """Measure inference time"""
    model.eval()
    
    # Warmup
    with torch.no_grad():
        for _ in range(num_warmup):
            _ = model(
                context=context,
                retrieved_seq=retrieved_seq,
                distances=distances
            )
    
    # Synchronize GPU
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Measure time
    times = []
    with torch.no_grad():
        for _ in range(num_runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start = time.time()
            
            _ = model(
                context=context,
                retrieved_seq=retrieved_seq,
                distances=distances
            )
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            end = time.time()
            times.append((end - start) * 1000)  # Convert to ms
    
    return np.mean(times), np.std(times), np.min(times), np.max(times)

In [20]:
#context.shape, retrieved_seq.shape, distances.shape

In [21]:
# Load or perform retrieval
if USE_ACTUAL_RETRIEVAL:
    try:
        retrieved_df = load_retrieved_data(
            DATASET_NAME, ROOT_DIR, CONTEXT_LENGTH, PREDICTION_LENGTH, metadata, RETRIEVE_SUFFIX
        )
        print(f"Using actual retrieved data from {DATASET_NAME}")
    except Exception as e:
        print(f"Failed to load retrieved data: {e}, using dummy data")
        retrieved_df = None
else:
    retrieved_df = None



Retrieved data not found at ../datasets/ETT-small/ETTh1_retrieve_ETTh1_512_only_self_train_None_X-norm.csv
Performing retrieval...
There are 7 variables in the original data
----------For one variable, we retrieve data from itself training set (X space with MinMax normalization)----------
----------Retrieving for variable: HUFL
Build X-space index (with MinMax normalization) with database: ['ETTh1_hour_512.pkl']
load database: ETTh1_hour_512.pkl
Database keys: ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL']...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66/66 [00:47<00:00,  1.40it/s]


----------Retrieving for variable: HULL
Build X-space index (with MinMax normalization) with database: ['ETTh1_hour_512.pkl']
load database: ETTh1_hour_512.pkl
Database keys: ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL']...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66/66 [00:47<00:00,  1.40it/s]


----------Retrieving for variable: MUFL
Build X-space index (with MinMax normalization) with database: ['ETTh1_hour_512.pkl']
load database: ETTh1_hour_512.pkl
Database keys: ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL']...


...

----------Retrieving for variable: OT
Build X-space index (with MinMax normalization) with database: ['ETTh1_hour_512.pkl']
load database: ETTh1_hour_512.pkl
Database keys: ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL']...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 66/66 [00:46<00:00,  1.41it/s]


Saving the retrieved data to ../datasets/ETT-small/ETTh1_retrieve_ETTh1_512_only_self_train_None_X-norm.csv
Using actual retrieved data from ETTh1

Processing model: cross_self
self.chronos_config.context_length 512
Total parameters: 52,297,857
Trainable parameters: 52,297,857
Failed to use retrieved data for cross_self: name 'create_inputs_from_retrieved_data' is not defined, using dummy data
Failed to use retrieved data for cross_self: name 'create_inputs_from_retrieved_data' is not defined, using dummy data
FLOPS measurement failed: 'function' object has no attribute 'training'


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Inference time: 9.663 ± 0.190 ms (min: 9.469, max: 10.650)

Processing model: cross
self.chronos_config.context_length 512
Total parameters: 52,297,857
Trainable parameters: 52,297,857
Failed to use retrieved data for cross: name 'create_inputs_from_retrieved_data' is not defined, using dummy data
Failed to use retrieved data for cross: name 'create_inputs_from_retrieved_data' is not defined, using dummy data
FLOPS measurement failed: 'function' object has no attribute 'training'
Inference time: 10.190 ± 1.171 ms (min: 9.084, max: 16.941)

Processing model: main_cross
self.chronos_config.context_length 512
Total parameters: 50,721,921
Trainable parameters: 50,721,921
Failed to use retrieved data for main_cross: name 'create_inputs_from_retrieved_data' is not defined, using dummy data
Failed to use retrieved data for main_cross: name 'create_inputs_from_retrieved_data' is not defined, using dummy data
FLOPS measurement failed: 'function' object has no attribute 'training'
Inference time

In [22]:
# Initialize models and measure
results = {}

for idx, (model_class, pipeline_class, model_name) in enumerate(zip(MODELS, PIPELINES, MODEL_NAMES)):
    print(f"\n{'='*60}")
    print(f"Processing model: {model_name}")
    print(f"{'='*60}")
    
    try:
        # Create model
        model = model_class(config=t5_config, augment=augment)
        model = model.to(device)
        model.eval()
        
        # Count parameters
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        
        # Create inputs (use actual retrieved data if available)
        if USE_ACTUAL_RETRIEVAL and retrieved_df is not None:
            try:
                context, retrieved_seq, distances = create_inputs_from_retrieved_data(
                    retrieved_df, 0, CONTEXT_LENGTH, NUM_RETRIEVED, PREDICTION_LENGTH, device, metadata, RETRIEVAL_DATABASE_DIR
                    )
                print(f"Using actual retrieved data for {model_name}")
            except Exception as e:
                print(f"Failed to use retrieved data for {model_name}: {e}, using dummy data")
                context, retrieved_seq, distances = create_dummy_inputs(
                    BATCH_SIZE, CONTEXT_LENGTH, NUM_RETRIEVED, RETRIEVED_SEQ_LENGTH
                    )
        else:
            context, retrieved_seq, distances = create_dummy_inputs(
                BATCH_SIZE, CONTEXT_LENGTH, NUM_RETRIEVED, RETRIEVED_SEQ_LENGTH
                )
        # Create inputs (use actual retrieved data if available)
        if USE_ACTUAL_RETRIEVAL and retrieved_df is not None:
            try:
                context, retrieved_seq, distances = create_inputs_from_retrieved_data(
                    retrieved_df, 0, CONTEXT_LENGTH, NUM_RETRIEVED, PREDICTION_LENGTH, device, metadata, RETRIEVAL_DATABASE_DIR
                    )
                print(f"Using actual retrieved data for {model_name}")
            except Exception as e:
                print(f"Failed to use retrieved data for {model_name}: {e}, using dummy data")
                context, retrieved_seq, distances = create_dummy_inputs(
                    BATCH_SIZE, CONTEXT_LENGTH, NUM_RETRIEVED, RETRIEVED_SEQ_LENGTH
                    )
        else:
            context, retrieved_seq, distances = create_dummy_inputs(
                BATCH_SIZE, CONTEXT_LENGTH, NUM_RETRIEVED, RETRIEVED_SEQ_LENGTH
                )
            BATCH_SIZE, CONTEXT_LENGTH, NUM_RETRIEVED, RETRIEVED_SEQ_LENGTH
#         )
        
        # Measure FLOPS
        flops = None
        if HAS_THOP:
            try:
                # Wrap model for thop (it needs a function that takes inputs)
                def model_forward(context, retrieved_seq, distances):
                    return model(
                        context=context,
                        retrieved_seq=retrieved_seq,
                        distances=distances
                    )
                
                flops, params = profile(
                    model_forward,
                    inputs=(context, retrieved_seq, distances),
                    verbose=False
                )
                flops_str = clever_format([flops], "%.3f")
                print(f"FLOPS: {flops_str}")
            except Exception as e:
                print(f"FLOPS measurement failed: {e}")
        else:
            print("FLOPS measurement skipped (thop not available)")
        
        # Measure inference time
        mean_time, std_time, min_time, max_time = measure_inference_time(
            model, context, retrieved_seq, distances
        )
        
        print(f"Inference time: {mean_time:.3f} ± {std_time:.3f} ms (min: {min_time:.3f}, max: {max_time:.3f})")
        
        # Store results
        results[model_name] = {
            'total_params': total_params,
            'trainable_params': trainable_params,
            'flops': flops,
            'mean_time_ms': mean_time,
            'std_time_ms': std_time,
            'min_time_ms': min_time,
            'max_time_ms': max_time
        }
        
        # Clean up
        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"Error processing {model_name}: {e}")
        import traceback
        traceback.print_exc()
        results[model_name] = {'error': str(e)}


Processing model: cross_self
self.chronos_config.context_length 512
Total parameters: 52,297,857
Trainable parameters: 52,297,857
Failed to use retrieved data for cross_self: name 'create_inputs_from_retrieved_data' is not defined, using dummy data
Failed to use retrieved data for cross_self: name 'create_inputs_from_retrieved_data' is not defined, using dummy data
FLOPS measurement failed: 'function' object has no attribute 'training'
Inference time: 10.144 ± 0.851 ms (min: 9.495, max: 11.870)

Processing model: cross
self.chronos_config.context_length 512
Total parameters: 52,297,857
Trainable parameters: 52,297,857
Failed to use retrieved data for cross: name 'create_inputs_from_retrieved_data' is not defined, using dummy data
Failed to use retrieved data for cross: name 'create_inputs_from_retrieved_data' is not defined, using dummy data
FLOPS measurement failed: 'function' object has no attribute 'training'
Inference time: 9.243 ± 0.167 ms (min: 9.089, max: 10.541)

Processing mo

In [ ]:
# Display results summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

import pandas as pd

summary_data = []
for model_name, result in results.items():
    if 'error' not in result:
        summary_data.append({
            'Model': model_name,
            'Parameters': f"{result['total_params']:,}",
            'FLOPS': f"{result['flops']:,}" if result['flops'] else 'N/A',
            'Mean Time (ms)': f"{result['mean_time_ms']:.3f}",
            'Std Time (ms)': f"{result['std_time_ms']:.3f}",
            'Min Time (ms)': f"{result['min_time_ms']:.3f}",
            'Max Time (ms)': f"{result['max_time_ms']:.3f}",
        })
    else:
        summary_data.append({
            'Model': model_name,
            'Parameters': 'ERROR',
            'FLOPS': 'ERROR',
            'Mean Time (ms)': 'ERROR',
            'Std Time (ms)': 'ERROR',
            'Min Time (ms)': 'ERROR',
            'Max Time (ms)': 'ERROR',
        })

df = pd.DataFrame(summary_data)
print(df.to_string(index=False))

In [ ]:
# Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Parameters comparison
if all('error' not in results.get(m, {}) for m in MODEL_NAMES):
    params = [results[m]['total_params'] for m in MODEL_NAMES if 'error' not in results.get(m, {})]
    model_names_valid = [m for m in MODEL_NAMES if 'error' not in results.get(m, {})]
    axes[0].bar(model_names_valid, params)
    axes[0].set_ylabel('Number of Parameters')
    axes[0].set_title('Model Parameters')
    axes[0].tick_params(axis='x', rotation=45)
    
    # FLOPS comparison
    if HAS_THOP and all(results.get(m, {}).get('flops') for m in model_names_valid):
        flops = [results[m]['flops'] for m in model_names_valid]
        axes[1].bar(model_names_valid, flops)
        axes[1].set_ylabel('FLOPS')
        axes[1].set_title('FLOPS')
        axes[1].tick_params(axis='x', rotation=45)
    
    # Inference time comparison
    mean_times = [results[m]['mean_time_ms'] for m in model_names_valid]
    std_times = [results[m]['std_time_ms'] for m in model_names_valid]
    axes[2].bar(model_names_valid, mean_times, yerr=std_times, capsize=5)
    axes[2].set_ylabel('Inference Time (ms)')
    axes[2].set_title('Inference Time')
    axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('flops_inference_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved as 'flops_inference_benchmark.png'")

In [ ]:
# Save results to file
import json

# Convert numpy types to native Python types for JSON serialization
def convert_to_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    return obj

serializable_results = convert_to_serializable(results)

with open('flops_inference_results.json', 'w') as f:
    json.dump(serializable_results, f, indent=2)

print("Results saved to 'flops_inference_results.json'")